In [2]:
import polars as pl
from pathlib import Path
ruta_base = Path.cwd().parent.parent
ruta_base

WindowsPath('C:/Users/ManuelOrellana/Desktop/Proyectos Aplicaciones/Aplicacion Prediccion Productos/Creacion de Modelos')

In [3]:
df = (
    pl.scan_csv(ruta_base/"Analisis y entrenamiento de modelos/Datasets/Modificados/train_rellenado.csv")
    .with_columns(pl.col("date").str.to_datetime().dt.date())
    .with_columns(pl.col("date").dt.year().alias("año"))
    .filter(pl.col("año") == 2017)
    .rename({
        "store_nbr": "numero_tienda",
        "item_nbr" : "numero_articulo",
        "unit_sales": "unidades_vendidas",
        "date": "fecha"
    })
    .sort("numero_tienda", "numero_articulo", "fecha")
    .with_columns(
        pl.col("unidades_vendidas")
        .shift(1)
        .over(["numero_tienda", "numero_articulo"])
        .fill_null(0)
        .alias("unidades_vendidas_ayer")
    )
    .with_columns(
        pl.col("unidades_vendidas")
        .shift(1)
        .rolling_mean(window_size=7, min_samples=1)
        .over(["numero_tienda", "numero_articulo"])
        .fill_null(0)
        .alias("unidades_vendidas_promedio_semanal")
    )
    .cast({
        "numero_tienda": pl.Int16,
        "numero_articulo": pl.Int32,
        "unidades_vendidas_ayer": pl.Float32,
        "unidades_vendidas_promedio_semanal" : pl.Float32,
    })
    .drop(["onpromotion", "id", "año"])
).collect(engine= "streaming")

In [4]:
df.head()

fecha,numero_tienda,numero_articulo,unidades_vendidas,unidades_vendidas_ayer,unidades_vendidas_promedio_semanal
date,i16,i32,f64,f32,f32
2017-01-02,1,96995,0.0,0.0,0.0
2017-01-03,1,96995,0.0,0.0,0.0
2017-01-04,1,96995,0.0,0.0,0.0
2017-01-05,1,96995,0.0,0.0,0.0
2017-01-06,1,96995,0.0,0.0,0.0


In [5]:
df = df.with_columns(pl.when(pl.col("unidades_vendidas") < 0).then(0).otherwise(pl.col("unidades_vendidas")).alias("unidades_vendidas"))

In [6]:
print(len(df.filter(pl.col("unidades_vendidas") < 0)))

0


In [7]:
df.write_parquet(ruta_base/"CreacionBaseConocimiento/BaseConocimiento/testeo_2017.parquet")